# Практическая работа №4: Элементы корреляционного анализа. Проверка статистической гипотезы о равенстве коэффициента корреляции нулю

Выполнили студенты гр. 2381 Ахметгареев Карим Ильгизович и Самулевич Степан Александрович. Вариант №1.

## Цель работы
Освоение основных понятий, связанных с корреляционной зависимостью между случайными величинами.

## Постановка задачи
Из заданной генеральной совокупности сформировать выборку по второму признаку. Провести статистическую обработку второй выборки в объёме практических работ №1 и №2, с целью определения точечных статистических оценок параметров распределения исследуемого признака (математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса и коэффициента вариации). Для системы двух случайных величин $X$ (первый признак) и $Y$ (второй признак) сформировать двумерную выборку и найти статистическую оценку коэффициента корреляции, построить доверительный интервал для коэффициента корреляции и осуществить проверку статистической гипотезы о равенстве коэффициента корреляции нулю. Полученные результаты содержательно проинтерпретировать.

## Основные теоретические положения

**Коэффициент корреляции Пирсона** – мера линейной связи между двумя случайными величинами. Выборочный коэффициент корреляции вычисляется по формуле:
$$ r_{xy} = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^n (x_i - \bar{x})^2 \sum_{i=1}^n (y_i - \bar{y})^2}}. $$

При группированных данных (интервальные ряды) используется метод условных вариант. Вводятся условные варианты:
$$ u_i = \frac{x_i - C_x}{h_x}, \quad v_j = \frac{y_j - C_y}{h_y}, $$
где $C_x$, $C_y$ – ложные нули (середины модальных интервалов), $h_x$, $h_y$ – ширина интервалов. Тогда коэффициент корреляции вычисляется через условные моменты.

**Доверительный интервал** для генерального коэффициента корреляции строится с использованием $z$-преобразования Фишера:
$$ z = \frac12 \ln\frac{1+r}{1-r}, \quad z \in N\left(\frac12\ln\frac{1+\rho}{1-\rho},\; \frac{1}{n-3}\right). $$
Интервал для $z$: $z \pm z_{\gamma/2}/\sqrt{n-3}$, затем обратное преобразование.

**Проверка гипотезы** $H_0:\rho=0$ осуществляется с помощью $t$-критерия:
$$ t = r \sqrt{\frac{n-2}{1-r^2}}, $$
который при справедливости $H_0$ имеет распределение Стьюдента с $n-2$ степенями свободы.

## Выполнение работы

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math

sns.set_theme(style="whitegrid")

# Загрузка данных
df = pd.read_csv('sample.csv', comment='#')
X = df['nu'].values
Y = df['E'].values
n = len(X)

### 1. Статистическая обработка второго признака Y (модуль упругости E)

In [73]:
k = int(np.ceil(1 + np.log2(n)))
y_min, y_max = Y.min(), Y.max()
h = (y_max - y_min) / k
bins = np.linspace(y_min, y_max, k+1)
mid = (bins[:-1] + bins[1:]) / 2
freq, _ = np.histogram(Y, bins=bins)

mean_y = np.sum(mid * freq) / n
var_y = np.sum(mid**2 * freq) / n - mean_y**2
sigma_y = np.sqrt(var_y)

m3 = np.sum((mid - mean_y)**3 * freq) / n
m4 = np.sum((mid - mean_y)**4 * freq) / n
asym_y = m3 / sigma_y**3
excess_y = m4 / sigma_y**4 - 3

mode_idx = np.argmax(freq)
f_m = freq[mode_idx]
f_m_prev = freq[mode_idx-1] if mode_idx > 0 else 0
f_m_next = freq[mode_idx+1] if mode_idx < k-1 else 0
Mo_y = bins[mode_idx] + h * ((f_m - f_m_prev) / ((f_m - f_m_prev) + (f_m - f_m_next)))

cum_freq = np.cumsum(freq)
med_idx = np.argmax(cum_freq >= n/2)
F_prev = cum_freq[med_idx-1] if med_idx > 0 else 0
Me_y = bins[med_idx] + h * ((n/2 - F_prev) / freq[med_idx])

cv_y = sigma_y / mean_y * 100

s2_y = n/(n-1) * var_y
s_y = np.sqrt(s2_y)

stats_y = pd.DataFrame({
    'Параметр': ['Объём выборки', 'Минимум', 'Максимум', 'Среднее', 'Дисперсия (смещ.)',
                 'СКО (смещ.)', 'Исправленная дисперсия', 'Исправленное СКО',
                 'Асимметрия', 'Эксцесс', 'Мода', 'Медиана', 'Коэф. вариации, %'],
    'Значение': [n, y_min, y_max, mean_y, var_y, sigma_y, s2_y, s_y, asym_y, excess_y,
                 Mo_y, Me_y, cv_y]
})

print("\nТаблица 1. Точечные оценки для второго признака Y (модуль упругости E)")
display(stats_y.round(4))


Таблица 1. Точечные оценки для второго признака Y (модуль упругости E)


,Параметр,Значение
0,Объём выборки,112.0000
1,Минимум,64.5000
2,Максимум,187.4000
3,Среднее,127.0473
4,Дисперсия (смещ.),588.8119
5,СКО (смещ.),24.2654
6,Исправленная дисперсия,594.1165
7,Исправленное СКО,24.3745
8,Асимметрия,-0.2672
9,Эксцесс,-0.1912


**Выводы по второму признаку:**
- Среднее значение модуля упругости $\bar{y} \approx 127$ кг/см².
- Коэффициент вариации $V \approx 19\%$, что указывает на условную однородность совокупности.
- Асимметрия $A_s \approx -0.27$ – распределение близко к симметричному, но имеется скошеность влево.
- Эксцесс $E \approx -0.2$ – распределение плосковершинное, но близко к нормальному.

### 2. Построение двумерного интервального вариационного ряда

In [74]:
# Используем то же число интервалов k для обоих признаков
# Границы для X (nu)
x_min, x_max = X.min(), X.max()
hx = (x_max - x_min) / k
bins_x = np.linspace(x_min, x_max, k+1)
mid_x = (bins_x[:-1] + bins_x[1:]) / 2

# Границы для Y (E)
y_min, y_max = Y.min(), Y.max()
hy = (y_max - y_min) / k
bins_y = np.linspace(y_min, y_max, k+1)
mid_y = (bins_y[:-1] + bins_y[1:]) / 2

# Двумерная гистограмма
H, xedges, yedges = np.histogram2d(X, Y, bins=[bins_x, bins_y])
H = H.T  # строки – Y, столбцы – X

df_corr = pd.DataFrame(H, index=np.round(mid_y, 2), columns=np.round(mid_x, 2))
df_corr.index.name = 'Y \ X'
df_corr.columns.name = 'X'

print("Таблица 2. Двумерный интервальный вариационный ряд (частоты n_ij)")
display(df_corr)

Таблица 2. Двумерный интервальный вариационный ряд (частоты n_ij)


<>:19: SyntaxWarning: invalid escape sequence '\ '
<>:19: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_16690/2854190588.py:19: SyntaxWarning: invalid escape sequence '\ '
  df_corr.index.name = 'Y \ X'


X,337.06,371.19,405.31,439.44,473.56,507.69,541.81,575.94
Y \ X,,,,,,,,
72.18,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
87.54,4.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
102.91,1.0,3.0,9.0,1.0,0.0,0.0,0.0,0.0
118.27,0.0,0.0,12.0,10.0,2.0,0.0,0.0,0.0
133.63,0.0,0.0,1.0,12.0,15.0,2.0,0.0,0.0
148.99,0.0,0.0,0.0,0.0,7.0,10.0,5.0,0.0
164.36,0.0,0.0,0.0,0.0,0.0,3.0,2.0,2.0
179.72,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0


### 3. Корреляционная таблица

In [75]:

mid_x_rounded = np.round(mid_x, 2)
mid_y_rounded = np.round(mid_y, 2)

corr_table = pd.DataFrame(H,
                          index=pd.Index(mid_y_rounded, name='Y \\ X'),
                          columns=pd.Index(mid_x_rounded, name='X'))
corr_table['n_y'] = n_y

total_row = pd.Series(np.append(n_x, n), index=corr_table.columns)
corr_table.loc['n_x'] = total_row

print("Таблица 3. Корреляционная таблица")
display(corr_table)

Таблица 3. Корреляционная таблица


X,337.06,371.19,405.31,439.44,473.56,507.69,541.81,575.94,n_y
Y \ X,,,,,,,,,
72.18,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
87.54,4.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,7.0
102.91,1.0,3.0,9.0,1.0,0.0,0.0,0.0,0.0,14.0
118.27,0.0,0.0,12.0,10.0,2.0,0.0,0.0,0.0,24.0
133.63,0.0,0.0,1.0,12.0,15.0,2.0,0.0,0.0,30.0
148.99,0.0,0.0,0.0,0.0,7.0,10.0,5.0,0.0,22.0
164.36,0.0,0.0,0.0,0.0,0.0,3.0,2.0,2.0,7.0
179.72,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,3.0
n_x,9.0,4.0,25.0,23.0,24.0,16.0,7.0,4.0,112.0


### 4. Вычисление выборочного коэффициента корреляции

#### Способ 1: стандартная формула по исходным данным

In [76]:

x_bar_grouped = np.sum(n_x * mid_x) / n
y_bar_grouped = np.sum(n_y * mid_y) / n

Sxx = np.sum(n_x * (mid_x - x_bar_grouped)**2)
Syy = np.sum(n_y * (mid_y - y_bar_grouped)**2)

Sxy = 0
for i in range(k):
    for j in range(k):
        Sxy += H[j, i] * (mid_x[i] - x_bar_grouped) * (mid_y[j] - y_bar_grouped)

r_pearson = Sxy / np.sqrt(Sxx * Syy)

print(f"Коэффициент корреляции (стандартная формула): {r_pearson:.6f}")

Коэффициент корреляции (стандартная формула): 0.903227


In [77]:
mode_idx_x = np.argmax(n_x)
Cx = mid_x[mode_idx_x]
mode_idx_y = np.argmax(n_y)
Cy = mid_y[mode_idx_y]
u = (mid_x - Cx) / hx
v = (mid_y - Cy) / hy
M_u = np.sum(n_x * u) / n
M_v = np.sum(n_y * v) / n
M_u2 = np.sum(n_x * u**2) / n
M_v2 = np.sum(n_y * v**2) / n
M_uv = 0
for i in range(k):
    for j in range(k):
        M_uv += H[j, i] * u[i] * v[j]
M_uv /= n
r_uv = (M_uv - M_u * M_v) / np.sqrt((M_u2 - M_u**2) * (M_v2 - M_v**2))

print(f"Коэффициент корреляции (условные варианты): r_uv = {r_uv:.6f}")
print(f"Разница между способами: {abs(r_pearson - r_uv):.2e}")

Коэффициент корреляции (условные варианты): r_uv = 0.903227
Разница между способами: 1.11e-16


Вычисленные двумя способами значения различаются на уровне машинной точности.

### 5. Доверительный интервал для коэффициента корреляции

Используем $z$-преобразование Фишера.

In [78]:
def fisher_z(r):
    return 0.5 * np.log((1+r)/(1-r))

def inv_fisher_z(z):
    return (np.exp(2*z) - 1) / (np.exp(2*z) + 1)

z = fisher_z(r_pearson)
se = 1 / np.sqrt(n - 3)

conf_levels = [0.95, 0.99]
z_crit = {0.95: 1.96, 0.99: 2.576}

print("Доверительные интервалы для генерального коэффициента корреляции:")
for gamma in conf_levels:
    z_low = z - z_crit[gamma] * se
    z_high = z + z_crit[gamma] * se
    r_low = inv_fisher_z(z_low)
    r_high = inv_fisher_z(z_high)
    print(f"γ = {gamma}: ({r_low:.4f}; {r_high:.4f})")

Доверительные интервалы для генерального коэффициента корреляции:
γ = 0.95: (0.8622; 0.9325)
γ = 0.99: (0.8462; 0.9398)


Доверительные интервалы не содержат нуль. Это говорит о том, что генеральный коэффициент корреляции значимо отличается от нуля.

### 6. Проверка гипотезы о равенстве коэффициента корреляции нулю

Проверим $H_0: \rho = 0$ против $H_1: \rho \neq 0$ при уровне значимости $\alpha = 0.05$.

In [79]:
alpha = 0.05
t_stat = r_pearson * np.sqrt((n-2) / (1 - r_pearson**2))
t_crit = stats.t.ppf(1 - alpha/2, df=n-2)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-2))

print(f"t-статистика: {t_stat:.4f}")
print(f"Критическое значение t_{alpha/2}({n-2}) = {t_crit:.4f}")
print(f"p-значение: {p_value:.6f}")

if abs(t_stat) > t_crit:
    print("Отвергаем H0: коэффициент корреляции значимо отличается от нуля.")
else:
    print("Нет оснований отвергнуть H0: коэффициент корреляции не является значимым.")

t-статистика: 22.0735
Критическое значение t_0.025(110) = 1.9818
p-значение: 0.000000
Отвергаем H0: коэффициент корреляции значимо отличается от нуля.


### Выводы

В ходе выполнения работы:
1. Проведена статистическая обработка второго признака (модуль упругости E). Получены точечные оценки параметров распределения.
2. Построен двумерный интервальный вариационный ряд и корреляционная таблица.
3. Вычислен выборочный коэффициент корреляции Пирсона $r = 0.903227$.
4. Построены доверительные интервалы для генерального коэффициента корреляции при надёжностях 0.95 и 0.99. Интервалы не содержат нуль.
5. Проверена гипотеза о равенстве коэффициента корреляции нулю. При $\alpha = 0.05$ нулевая гипотеза отвергается. Это свидетельствует о наличии статистически значимой линейной связи между объёмным весом древесины и модулем упругости, что соответствует и физической интуиции.

Таким образом, цели работы достигнуты, основные понятия корреляционного анализа освоены.